In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import pandas as pd
import numpy as np
from PIL import Image
import os
import cv2
from pathlib import Path
from typing import Union, List, Dict, Tuple
import warnings
from torch.utils.data import Dataset, DataLoader, random_split
import glob
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
models_to_compare = {
    # ResNet Family
      "ResNet50": "resnet50",
      "ResNeXt50": "resnext50",

      # EfficientNet Family
      "EfficientNet-B1": "efficientnet_b1",
      "EfficientNet-V2-S": "efficientnet_v2_s",

      # DenseNet Family
      "DenseNet121": "densenet121",

      # MobileNet Family
      "MobileNet-V3-Large": "mobilenet_v3_large",
    # ConvNeXt Family
    "ConvNeXt-Small": "convnext_small",

    # Vision Transformers
    "ViT-B/16": "vit_b_16",
    "Swin-T": "swin_t",
    "MaxViT-T": "maxvit_t",

    # RegNet Family
    "RegNet-Y-800MF": "regnet_y_800mf"
}

In [ ]:

# --- Custom Dataset Class for Classification ---
NUM_CLASSES = 3  # Original number of classes (0–7)
BIN_MAPPING = {
    0: 0,        # bin1
    1: 0, 2: 0,  # bin2
    3: 0, 4: 1, 5: 1, 6: 2, 7: 2, 8:2  # bin3
}

class ImageClassificationDataset(Dataset):
    def __init__(self, image_folder, csv_file, transform=None, target_col="Index"):
        assert len(image_folder) == len(csv_file), "image_folders and csv_files must have the same length"

        self.transform = transform or transforms.ToTensor()
        self.image_paths = []
        self.targets = []

        for img_folder, csv_path in zip(image_folder, csv_file):
            df = pd.read_csv(csv_path)
            df['PLOT'] = pd.to_numeric(df['PLOT'], errors='coerce')
            df = df.replace([float('inf'), float('-inf')], pd.NA)
            df = df.dropna(subset=['PLOT'])

            df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
            df = df.replace([float('inf'), float('-inf')], pd.NA)
            df = df.dropna(subset=[target_col])

            df['PLOT'] = df['PLOT'].astype(int)
            df[target_col] = df[target_col].astype(float)

            # Convert Index values (1.0–8.0) to original class labels (0–7)
            df['class_label'] = (df[target_col] - 1).astype(int)

            # Filter out invalid classes
            valid_mask = (df['class_label'] >= 0) & (df['class_label'] < 9)
            df = df[valid_mask]

            # Apply binning
            df['binned_label'] = df['class_label'].map(BIN_MAPPING)

            id_to_target = dict(zip(df['PLOT'], df['binned_label']))
            image_files = glob.glob(os.path.join(img_folder, "*.png"))

            for img_path in image_files:
                filename = os.path.basename(img_path)
                try:
                    plot_id = int(filename.split('_')[-1].split('.')[0])
                except Exception as e:
                    continue

                if plot_id in id_to_target:
                    self.image_paths.append(img_path)
                    self.targets.append(id_to_target[plot_id])

            print(f"✅ Loaded {len(image_files)} images from {img_folder}")

        print(f"✅ Loaded {len(self.image_paths)} total images from {len(image_folder)} folders.")

        # Print class distribution after binning
        unique, counts = np.unique(self.targets, return_counts=True)
        print(f"\n📊 Binned Class Distribution:")
        for cls, cnt in zip(unique, counts):
            print(f"   Bin {cls+1}: {cnt} samples ({cnt/len(self.targets)*100:.1f}%)")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        target = self.targets[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error opening image {img_path}: {e}. Skipping.")
            return None

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(target, dtype=torch.long)


In [ ]:

# --- Model Building Function for Classification ---
def get_classification_model(model_name, num_classes=NUM_CLASSES, fine_tune_mode="full"):
    """Get a pretrained model for classification"""
    model = None

    # ResNet Family
    if model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == "resnext50":
        model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == "resnext101":
        model = models.resnext101_32x8d(weights=models.ResNeXt101_32X8D_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)

    # EfficientNet Family
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "efficientnet_b1":
        model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "efficientnet_b2":
        model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "efficientnet_v2_s":
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "efficientnet_v2_m":
        model = models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    # DenseNet Family
    elif model_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == "densenet161":
        model = models.densenet161(weights=models.DenseNet161_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == "densenet169":
        model = models.densenet169(weights=models.DenseNet169_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    # MobileNet Family
    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif model_name == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    # ConvNeXt Family
    elif model_name == "convnext_tiny":
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    elif model_name == "convnext_small":
        model = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    elif model_name == "convnext_base":
        model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.DEFAULT)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

    # Vision Transformers
    elif model_name == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif model_name == "vit_b_32":
        model = models.vit_b_32(weights=models.ViT_B_32_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif model_name == "vit_l_16":
        model = models.vit_l_16(weights=models.ViT_L_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif model_name == "swin_t":
        model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif model_name == "swin_s":
        model = models.swin_s(weights=models.Swin_S_Weights.DEFAULT)
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif model_name == "swin_b":
        model = models.swin_b(weights=models.Swin_B_Weights.DEFAULT)
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif model_name == "maxvit_t":
        model = models.maxvit_t(weights=models.MaxVit_T_Weights.DEFAULT)
        model.classifier[5] = nn.Linear(model.classifier[5].in_features, num_classes)

    # RegNet Family
    elif model_name == "regnet_y_400mf":
        model = models.regnet_y_400mf(weights=models.RegNet_Y_400MF_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "regnet_y_800mf":
        model = models.regnet_y_800mf(weights=models.RegNet_Y_800MF_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "regnet_y_1_6gf":
        model = models.regnet_y_1_6gf(weights=models.RegNet_Y_1_6GF_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    else:
        raise ValueError(f"Invalid model name: {model_name}")

    # Apply fine-tuning strategy
    if fine_tune_mode == "freeze":
        for param in model.parameters():
            param.requires_grad = False
        if hasattr(model, 'fc'):
            for param in model.fc.parameters():
                param.requires_grad = True
        elif hasattr(model, 'classifier'):
            for param in model.classifier.parameters():
                param.requires_grad = True
        elif hasattr(model, 'head'):
            for param in model.head.parameters():
                param.requires_grad = True
        elif hasattr(model, 'heads'):
            for param in model.heads.parameters():
                param.requires_grad = True
    elif fine_tune_mode == "partial":
        total_layers = len(list(model.parameters()))
        layers_to_freeze = total_layers // 2
        for i, param in enumerate(model.parameters()):
            param.requires_grad = i >= layers_to_freeze

    return model

# AgriBlast Model Inference


In [ ]:
# ---- Inference configuration (update these) ----
MODEL_PATH = "/content/drive/MyDrive/EfficientNet-V2-S.pth"
MODEL_KEY = "efficientnet_v2_s"  # Architecture key used while training
INFER_IMAGE_FOLDER = [
        "/content/drive/MyDrive/bl1_s1_b3_2025/png",
        "/content/drive/MyDrive/bl1_s1_b4_2025/png",
        "/content/drive/MyDrive/bl1_s2_b3_2025/png",
        "/content/drive/MyDrive/bl1_s2_b4_2025/png",
        "/content/drive/MyDrive/bl1_s3_b3_2025/png",
        "/content/drive/MyDrive/bl1_s3_b4_2025/png",
        "/content/drive/MyDrive/bl2_s1_b3_2025/png",
        "/content/drive/MyDrive/bl2_s1_b4_2025/png",
        "/content/drive/MyDrive/bl2_s2_b3_2025/png",
        "/content/drive/MyDrive/bl2_s2_b4_2025/png",
        "/content/drive/MyDrive/bl2_s3_b3_2025/png",
        "/content/drive/MyDrive/bl2_s3_b4_2025/png",

                     ]  # str or list[str]
INFER_CSV_FILE = [
        "/content/drive/MyDrive/b3_bl1_2025.csv",
        "/content/drive/MyDrive/b4_bl1_2025.csv",
        "/content/drive/MyDrive/b3_bl1_2025.csv",
        "/content/drive/MyDrive/b4_bl1_2025.csv",
        "/content/drive/MyDrive/b3_bl1_2025.csv",
        "/content/drive/MyDrive/b4_bl1_2025.csv",
        "/content/drive/MyDrive/b3_bl2_2025.csv",
        "/content/drive/MyDrive/b4_bl2_2025.csv",
        "/content/drive/MyDrive/b3_bl2_2025.csv",
        "/content/drive/MyDrive/b4_bl2_2025.csv",
        "/content/drive/MyDrive/b3_bl2_2025.csv",
        "/content/drive/MyDrive/b4_bl2_2025.csv",
        ]         # str or list[str]
TARGET_COL = "Index"
INFER_BATCH_SIZE = 64
PREDICTIONS_OUT_CSV = "inference_predictions.csv"

In [ ]:
# ==================== INFERENCE SCRIPT ====================

def _as_list(x):
    return x if isinstance(x, (list, tuple)) else [x]


# Keep inference preprocessing identical to training preprocessing
inference_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class InferenceImageClassificationDataset(ImageClassificationDataset):
    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        if item is None:
            return None
        image, target = item
        return image, target, self.image_paths[idx]


def collate_fn_inference(batch):
    batch = [x for x in batch if x is not None]
    if not batch:
        return None, None, None
    images, targets, paths = zip(*batch)
    return torch.stack(images), torch.stack(targets), list(paths)


def load_weights_safely(model, checkpoint_path, map_location):
    checkpoint = torch.load(checkpoint_path, map_location=map_location)

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    cleaned_state_dict = {}
    for key, value in state_dict.items():
        cleaned_key = key.replace("module.", "", 1) if key.startswith("module.") else key
        cleaned_state_dict[cleaned_key] = value

    model.load_state_dict(cleaned_state_dict, strict=True)
    return model


def run_inference_with_metrics(model_path, model_key, image_folder, csv_file, target_col="Index",
                               batch_size=64, predictions_out_csv="inference_predictions.csv"):
    image_folder = _as_list(image_folder)
    csv_file = _as_list(csv_file)

    if len(image_folder) != len(csv_file):
        raise ValueError("image_folder and csv_file must have the same length.")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model weights not found: {model_path}")

    if "models_to_compare" in globals() and model_key in models_to_compare:
        model_key = models_to_compare[model_key]

    dataset = InferenceImageClassificationDataset(
        image_folder=image_folder,
        csv_file=csv_file,
        transform=inference_transform,
        target_col=target_col
    )

    if len(dataset) == 0:
        raise ValueError("No valid samples found. Check image folder, CSV format, and filename pattern.")

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn_inference
    )

    model = get_classification_model(model_key, num_classes=NUM_CLASSES, fine_tune_mode="full")
    model = load_weights_safely(model, model_path, map_location=device)
    model = model.to(device)
    model.eval()

    all_targets, all_preds, all_probs, all_paths = [], [], [], []

    with torch.no_grad():
        for images, targets, paths in loader:
            if images is None:
                continue

            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_targets.extend(targets.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_paths.extend(paths)

    if len(all_targets) == 0:
        raise RuntimeError("No samples were evaluated. Verify your data files.")

    all_probs_np = np.array(all_probs)
    confidence = all_probs_np.max(axis=1)

    metrics = {
        "accuracy": accuracy_score(all_targets, all_preds),
        "f1_weighted": f1_score(all_targets, all_preds, average="weighted", zero_division=0),
        "precision_weighted": precision_score(all_targets, all_preds, average="weighted", zero_division=0),
        "recall_weighted": recall_score(all_targets, all_preds, average="weighted", zero_division=0),
        "confusion_matrix": confusion_matrix(all_targets, all_preds, labels=list(range(NUM_CLASSES))),
        "classification_report": classification_report(
            all_targets,
            all_preds,
            labels=list(range(NUM_CLASSES)),
            target_names=[f"Class {i}" for i in range(NUM_CLASSES)],
            zero_division=0
        )
    }

    preds_df = pd.DataFrame({
        "image_path": all_paths,
        "true_label": all_targets,
        "pred_label": all_preds,
        "confidence": confidence,
    })

    for cls_idx in range(NUM_CLASSES):
        preds_df[f"prob_class_{cls_idx}"] = all_probs_np[:, cls_idx]

    preds_df.to_csv(predictions_out_csv, index=False)

    print("\\nInference Metrics")
    print(f"Accuracy            : {metrics['accuracy']:.4f}")
    print(f"F1 (weighted)       : {metrics['f1_weighted']:.4f}")
    print(f"Precision (weighted): {metrics['precision_weighted']:.4f}")
    print(f"Recall (weighted)   : {metrics['recall_weighted']:.4f}")
    print("\\nClassification Report")
    print(metrics["classification_report"])
    print(f"\\nPredictions saved to: {predictions_out_csv}")

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        metrics["confusion_matrix"],
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[f"Pred {i}" for i in range(NUM_CLASSES)],
        yticklabels=[f"True {i}" for i in range(NUM_CLASSES)],
    )
    plt.title("Inference Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

    return metrics, preds_df


# Run inference
inference_metrics, inference_predictions = run_inference_with_metrics(
    model_path=MODEL_PATH,
    model_key=MODEL_KEY,
    image_folder=INFER_IMAGE_FOLDER,
    csv_file=INFER_CSV_FILE,
    target_col=TARGET_COL,
    batch_size=INFER_BATCH_SIZE,
    predictions_out_csv=PREDICTIONS_OUT_CSV,
)

inference_predictions.head()
print(inference_metrics)


✅ Loaded 1660 images from /content/drive/MyDrive/bl1_s1_b3_2025/png
✅ Loaded 624 images from /content/drive/MyDrive/bl1_s1_b4_2025/png
✅ Loaded 1660 images from /content/drive/MyDrive/bl1_s2_b3_2025/png
✅ Loaded 624 images from /content/drive/MyDrive/bl1_s2_b4_2025/png
✅ Loaded 1660 images from /content/drive/MyDrive/bl1_s3_b3_2025/png
✅ Loaded 624 images from /content/drive/MyDrive/bl1_s3_b4_2025/png
✅ Loaded 1660 images from /content/drive/MyDrive/bl2_s1_b3_2025/png
✅ Loaded 625 images from /content/drive/MyDrive/bl2_s1_b4_2025/png
✅ Loaded 1660 images from /content/drive/MyDrive/bl2_s2_b3_2025/png
✅ Loaded 624 images from /content/drive/MyDrive/bl2_s2_b4_2025/png
✅ Loaded 1660 images from /content/drive/MyDrive/bl2_s3_b3_2025/png
✅ Loaded 624 images from /content/drive/MyDrive/bl2_s3_b4_2025/png
✅ Loaded 13680 total images from 12 folders.

📊 Binned Class Distribution:
   Bin 1: 3738 samples (27.3%)
   Bin 2: 6735 samples (49.2%)
   Bin 3: 3207 samples (23.4%)
Downloading: "https://

100%|██████████| 82.7M/82.7M [00:00<00:00, 156MB/s]
